# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [1]:
import re
import math
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import gutenberg
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("gutenberg", quiet=True)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

def preprocess(text):
    tokens = re.findall(r"[a-z]+", text.lower())
    result = []
    for t in tokens:
        if len(t) > 2 and t not in ENGLISH_STOP_WORDS:
            result.append(t)
    return result

ids = []
titles = []
texts = []
tokens = []
for i, f in enumerate(gutenberg.fileids()):
    raw = gutenberg.raw(f)
    ids.append("D" + str(i + 1).zfill(2))
    titles.append(f.replace(".txt", ""))
    texts.append(raw)
    tokens.append(preprocess(raw))

corpus = pd.DataFrame({"doc_id": ids, "title": titles, "text": texts, "tokens": tokens})
print(corpus[["doc_id", "title"]])

N = len(corpus)
DF = {}
for toks in corpus["tokens"]:
    for t in set(toks):
        DF[t] = DF.get(t, 0) + 1

IDF = {}
for t in DF:
    IDF[t] = math.log((N - DF[t] + 0.5) / (DF[t] + 0.5) + 1)

print("Documentos:", N)
print("Vocabulario:", len(DF))

   doc_id                title
0     D01          austen-emma
1     D02    austen-persuasion
2     D03         austen-sense
3     D04            bible-kjv
4     D05          blake-poems
5     D06       bryant-stories
6     D07  burgess-busterbrown
7     D08        carroll-alice
8     D09      chesterton-ball
9     D10     chesterton-brown
10    D11  chesterton-thursday
11    D12    edgeworth-parents
12    D13   melville-moby_dick
13    D14      milton-paradise
14    D15   shakespeare-caesar
15    D16   shakespeare-hamlet
16    D17  shakespeare-macbeth
17    D18       whitman-leaves
Documentos: 18
Vocabulario: 41071


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [2]:
queries = [
    ("whale", "Información sobre ballenas."),
    ("king", "Pasajes sobre reyes."),
    ("love marriage", "Historias de amor y matrimonio."),
    ("god lord", "Textos religiosos sobre Dios y el Señor."),
    ("alice rabbit hole", "La escena de Alicia y el conejo."),
    ("whale hunting ship sea captain", "Caza de ballenas y vida a bordo."),
    ("murder king throne betrayal", "Tramas de poder, asesinato y traición."),
    ("xyzzy quantum", "Consulta sin términos del vocabulario."),
]

doc_len = []
tf_docs = []
for toks in corpus["tokens"]:
    doc_len.append(len(toks))
    tf = {}
    for t in toks:
        tf[t] = tf.get(t, 0) + 1
    tf_docs.append(tf)
avgdl = np.mean(doc_len)

def same_tokens(tokens):
    return tokens

vectorizer = TfidfVectorizer(analyzer=same_tokens, use_idf=True, smooth_idf=False, norm=None)
X = vectorizer.fit_transform(list(corpus["tokens"]))

def bm25_term(t, j, k1, b):
    f = tf_docs[j].get(t, 0)
    if f == 0 or t not in IDF:
        return 0.0
    denom = f + k1 * (1 - b + b * doc_len[j] / avgdl)
    return IDF[t] * f * (k1 + 1) / denom

def bm25_scores(q_tokens, k1=1.2, b=0.75):
    scores = np.zeros(N)
    for t in set(q_tokens):
        for j in range(N):
            scores[j] += bm25_term(t, j, k1, b)
    return scores

def tfidf_scores(q_tokens):
    q = vectorizer.transform([q_tokens])
    if q.nnz == 0:
        return np.zeros(N)
    return cosine_similarity(q, X)[0]

def sort_key(item):
    return (-round(item[1], 12), item[0])

def rank(scores, top=10):
    items = []
    for j in range(N):
        if scores[j] > 0:
            items.append((ids[j], float(scores[j])))
    items.sort(key=sort_key)
    return items[:top]

q_tokens_list = []
res_tfidf = []
res_bm25 = []
for text, objetivo in queries:
    toks = preprocess(text)
    q_tokens_list.append(toks)
    res_tfidf.append(rank(tfidf_scores(toks)))
    res_bm25.append(rank(bm25_scores(toks)))

    print("Consulta:", text, "-", objetivo)
    print("df:", {t: DF.get(t, 0) for t in toks})
    print("TF-IDF:", len(res_tfidf[-1]), "documentos | BM25:", len(res_bm25[-1]), "documentos")
    print(pd.DataFrame(res_tfidf[-1], columns=["doc_id", "coseno"]))
    print(pd.DataFrame(res_bm25[-1], columns=["doc_id", "bm25"]))
    print()

Consulta: whale - Información sobre ballenas.
df: {'whale': 6}
TF-IDF: 6 documentos | BM25: 6 documentos
  doc_id    coseno
0    D13  0.520956
1    D06  0.026811
2    D18  0.005134
3    D16  0.002965
4    D09  0.001359
5    D04  0.000198
  doc_id      bm25
0    D13  2.355879
1    D06  2.267835
2    D18  1.853818
3    D16  1.845304
4    D09  1.610523
5    D04  0.762325

Consulta: king - Pasajes sobre reyes.
df: {'king': 15}
TF-IDF: 10 documentos | BM25: 10 documentos
  doc_id    coseno
0    D16  0.143664
1    D06  0.133110
2    D04  0.094549
3    D17  0.065795
4    D08  0.054534
5    D14  0.031688
6    D10  0.016970
7    D13  0.015321
8    D05  0.004961
9    D12  0.004911
  doc_id      bm25
0    D04  0.446813
1    D16  0.446470
2    D06  0.445866
3    D08  0.444441
4    D17  0.444184
5    D14  0.438450
6    D13  0.434074
7    D10  0.430254
8    D09  0.409788
9    D12  0.406775

Consulta: love marriage - Historias de amor y matrimonio.
df: {'love': 15, 'marriage': 12}
TF-IDF: 10 document

**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [3]:
title_of = dict(zip(ids, titles))

def label(doc_id):
    return doc_id + " " + title_of[doc_id]

for i in range(len(queries)):
    T = res_tfidf[i]
    B = res_bm25[i]
    filas = []
    for pos in range(10):
        fila = {"Posición": pos + 1, "ID y título TF-IDF": "", "Coseno": "", "ID y título BM25": "", "Score BM25": ""}
        if pos < len(T):
            fila["ID y título TF-IDF"] = label(T[pos][0])
            fila["Coseno"] = round(T[pos][1], 4)
        if pos < len(B):
            fila["ID y título BM25"] = label(B[pos][0])
            fila["Score BM25"] = round(B[pos][1], 4)
        filas.append(fila)

    set_t = set([x[0] for x in T])
    set_b = set([x[0] for x in B])
    print("Consulta:", queries[i][0])
    print(pd.DataFrame(filas))
    print("Documentos en común:", len(set_t & set_b))
    print("Tamaño TF-IDF:", len(T), "| Tamaño BM25:", len(B))
    if len(T) > 0:
        print("Primero TF-IDF:", T[0][0], "| Primero BM25:", B[0][0])
    print()

qi = 2
q_toks = q_tokens_list[qi]
pos_t = {}
for p, x in enumerate(res_tfidf[qi]):
    pos_t[x[0]] = p + 1
pos_b = {}
for p, x in enumerate(res_bm25[qi]):
    pos_b[x[0]] = p + 1

cambios = []
for d in pos_b:
    if d in pos_t:
        cambios.append((abs(pos_t[d] - pos_b[d]), d))
cambios.sort(reverse=True)
elegidos = [x[1] for x in cambios[:2]]

print("Consulta analizada:", queries[qi][0])
for d in elegidos:
    j = ids.index(d)
    print(label(d), "| posición TF-IDF:", pos_t[d], "| posición BM25:", pos_b[d])
    filas = []
    for t in q_toks:
        filas.append({
            "Término": t,
            "|d|": doc_len[j],
            "f(t,d)": tf_docs[j].get(t, 0),
            "df(t)": DF.get(t, 0),
            "Aporte BM25": round(bm25_term(t, j, 1.2, 0.75), 4),
        })
    print(pd.DataFrame(filas))
    print("Longitud media:", round(avgdl, 1))
    print()

Consulta: whale
   Posición      ID y título TF-IDF  Coseno        ID y título BM25 Score BM25
0         1  D13 melville-moby_dick   0.521  D13 melville-moby_dick     2.3559
1         2      D06 bryant-stories  0.0268      D06 bryant-stories     2.2678
2         3      D18 whitman-leaves  0.0051      D18 whitman-leaves     1.8538
3         4  D16 shakespeare-hamlet   0.003  D16 shakespeare-hamlet     1.8453
4         5     D09 chesterton-ball  0.0014     D09 chesterton-ball     1.6105
5         6           D04 bible-kjv  0.0002           D04 bible-kjv     0.7623
6         7                                                                   
7         8                                                                   
8         9                                                                   
9        10                                                                   
Documentos en común: 6
Tamaño TF-IDF: 6 | Tamaño BM25: 6
Primero TF-IDF: D13 | Primero BM25: D13

Consulta: king
  

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [4]:
q_text = "whale hunting ship sea captain"
q_toks = preprocess(q_text)

configs = [("Base", 1.2, 0.75), ("Sin ajuste por longitud", 1.2, 0.0), ("Mayor efecto de repeticiones", 2.0, 0.75)]

todos = []
for nombre, k1, b in configs:
    s = bm25_scores(q_toks, k1, b)
    r = rank(s, top=N)
    posiciones = {}
    for p, x in enumerate(r):
        posiciones[x[0]] = p + 1
    todos.append((posiciones, s))

base_top = [x[0] for x in rank(bm25_scores(q_toks, 1.2, 0.75), top=10)]

filas = []
for d in base_top:
    j = ids.index(d)
    fila = {"Documento": label(d), "|d|": doc_len[j]}
    for k in range(len(configs)):
        fila["Pos " + configs[k][0]] = todos[k][0].get(d, "-")
        fila["Score " + configs[k][0]] = round(todos[k][1][j], 3)
    filas.append(fila)
print("Consulta:", q_text, "| longitud media:", round(avgdl, 1))
print(pd.DataFrame(filas).to_string())

for d in base_top[:3]:
    j = ids.index(d)
    filas = []
    for t in q_toks:
        fila = {"Término": t, "f(t,d)": tf_docs[j].get(t, 0)}
        for nombre, k1, b in configs:
            fila[nombre] = round(bm25_term(t, j, k1, b), 3)
        filas.append(fila)
    print()
    print("Aportes por término en", label(d))
    print(pd.DataFrame(filas))

Consulta: whale hunting ship sea captain | longitud media: 49592.3
                 Documento     |d|  Pos Base  Score Base  Pos Sin ajuste por longitud  Score Sin ajuste por longitud  Pos Mayor efecto de repeticiones  Score Mayor efecto de repeticiones
0   D13 melville-moby_dick   95943         1       5.830                            1                          5.911                                 1                               7.799
1       D06 bryant-stories   19222         2       5.363                            3                          4.938                                 2                               6.847
2       D18 whitman-leaves   58668         3       5.083                            2                          5.175                                 3                               6.340
3            D04 bible-kjv  329877         4       3.484                            4                          4.937                                 4                               4.19